In [1]:
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from tqdm import tqdm
import numpy as np

from datasets.bcdata import BCDataDataset, collate_heatmap_points
from datasets.transforms import PointsToLocalizationHeatmap, PointsToCountHeatmap


from visualization import overlay_heatmap
from training import train


from models.models import HybridModel
from models.losses import weighted_sigmoid_mse_from_logits, softplus_mse_from_logits, l1_count_from_density_logits

from utils.debug import print_info



import torch
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


In [2]:
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)


data_root = Path(cfg["h200_paths"]["data_root"])
checkpoint_dir = Path(cfg["h200_paths"]["checkpoint_dir"])

print(f"data_root: {data_root}")
print(f"checkpoint_dir: {checkpoint_dir}")

data_root: /raid/datasets/Yeldos/BCData
checkpoint_dir: checkpoints


In [3]:
loc_heatmap_generator = PointsToLocalizationHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)
count_heatmap_generator = PointsToCountHeatmap(out_hw=(160,160), in_hw=(640,640), sigma=2.0)

dataset = BCDataDataset(root = data_root,
                        split="train",
                        target_loc_transform = loc_heatmap_generator,
                        target_count_transform = count_heatmap_generator)

In [4]:
train_loader = DataLoader(
    dataset,
    batch_size=8,        # choose based on GPU memory (640×640 images are large)
    shuffle=True,
    num_workers=2,       # use 0 if debugging
    pin_memory=True,     # recommended when using GPU
    drop_last=True,       # optional, useful for BatchNorm
    collate_fn = collate_heatmap_points
)

In [5]:
model = HybridModel()
device = 'cuda'
model.to(device);

optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-2
        )



In [ ]:
m_max = 0.174835
m_dens = 5.013973e-05
m_count = 59.171053

eps = 1e-8
lambda_dens = m_max / (m_dens + eps)
lambda_count = m_max / (m_count + eps)


In [18]:
print(f"lambda_dens   {lambda_dens}")
print(f"lambda_count  {lambda_count}")

lambda_dens   3486.2600456672444
lambda_count  0.002954738695802027


In [8]:
Lmax_list = []
Ldens_list = []
Lcount_list = []

In [22]:
model.train()

for img, loc_heatmap, count_heatmap, pos_pts, neg_pts in tqdm(train_loader):


    img = img.to(device)
    loc_heatmap = loc_heatmap.to(device)
    count_heatmap = count_heatmap.to(device)

    pred_loc_hm, pred_den_hm, pred_count = model(img)
    pred_den_hm = pred_den_hm.to(device)

    Lmax = weighted_sigmoid_mse_from_logits(pred_logits = pred_loc_hm, target = loc_heatmap)
    Ldens = softplus_mse_from_logits(pred_logits = pred_den_hm, target = count_heatmap)

    gtN = [(len(tmp1), len(tmp2)) for tmp1, tmp2 in zip(pos_pts, neg_pts)]  # (B, 2)

    gtN = torch.tensor(gtN)
    gtN = gtN.to(device)

    Lcount = l1_count_from_density_logits(pred_logits = pred_den_hm, gtN = gtN)

    loss = Lmax + lambda_dens * Ldens + lambda_count * Lcount


    Lmax_list.append(Lmax.item())
    Ldens_list.append(Ldens.item())
    Lcount_list.append(Lcount.item())

    loss.backward()
    optimizer.step()




100%|██████████| 100/100 [00:36<00:00,  2.72it/s]


In [21]:
print(np.mean(Lmax_list[5:]))
print(lambda_dens * np.mean(Ldens_list[5:]))
print(lambda_count * np.mean(Lcount_list[5:]))

0.12330953094520067
0.17337923483404666
0.17355590805170826


In [ ]:
# losses = train(model=model, num_epochs=2, train_loader=train_loader, val_loader=None, loss_function=heatmap_weighted_mse_loss, count_metrics=count_metrics, forplot_img=None)